# Model Training

1. Hyperparameters Selection (random search):
    - For each sample, *nsubset* subets are generated. Each subset has *ncell* elements (cells)
    - A number of filters is randomly chosen
    - One at time the % of top-k pooling is chosen
2. Training:
    - The model is trained for *nrun* times (each run with a randomly chosen selection of hyperparameters) and it is tested on the validation set
    - After each run, the model stores hyperparameters, weights and model performance metrics.
    - Finally the 3 combination of hyperparameters and the weights that performed the best, are then chosed for final testing on test sataset
3. Testing:
    - The 3 best combinations are then used to perform the prediction on the Test dataset
    - The result is then the mean performance of the 3
    - The model that performed the best is then use to determine the number of filter to be use and perform clustering





# Model Architecture:

- Input Layer -> Takes data and returns a matrix of (number of cells) x (number of marker)

- Convlutional Layer -> A matrix of dimension (number of cells) x (number of filters) is generated, where each row represents a cell and each column the linear cmbination of the markers values and finally applies RELU

- Top-k Pooling Layer -> For each filter, only the most reactive k cells are chosen. Compute the mean of the top-k activations for each filter

- Output Layer -> This layer produces an output for every class (phenotypes)

- Optimization -> loss function is minimized, tuning the weights


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import shutil
import pandas as pd
import random
import copy
import numpy as np # Import numpy for checking finite values
import matplotlib.pyplot as plt # Import matplotlib for potential debugging
import os
import math # Import math for ceil
from sklearn.manifold import TSNE # Import TSNE to check default perplexity
import sys


In [3]:

utils_path = '/content/drive/MyDrive/Master Thesis'
sys.path.append(utils_path)

from Dataset_Elaboration_Class import *


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:

fixed_path = '/content/drive/MyDrive/Master Thesis/CellCNN/'
cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'


In [5]:
#!pip install git+https://github.com/eiriniar/CellCnn.git
# Installa tutte le dipendenze necessarie per CellCNN
!pip install flowio
!pip install tensorflow


In [6]:
#!wget https://raw.githubusercontent.com/eiriniar/CellCnn/python3/cellCnn/utils.py
#!wget https://raw.githubusercontent.com/eiriniar/CellCnn/python3/cellCnn/downsample.py
#!wget https://raw.githubusercontent.com/eiriniar/CellCnn/python3/cellCnn/__init__.py -O original_init.py

!mkdir -p /content/cellCnn # creates a folder

# Move downloaded files
shutil.copy2(f'{fixed_path}utils.py', 'cellCnn/utils.py')
shutil.copy2(f'{fixed_path}downsample.py', 'cellCnn/downsample.py')

# Copy fixed files
shutil.copy2(f'{fixed_path}model.py', 'cellCnn/model.py')
shutil.copy2(f'{fixed_path}plotting.py', 'cellCnn/plotting.py')

# create an __init__.py without theano
with open('cellCnn/__init__.py', 'w') as f:
    f.write('''# Clean CellCNN without Theano dependencies
from .model import CellCnn''')

# import downloaded modules
from cellCnn.model import CellCnn
import cellCnn.plotting as plotting

import cellCnn.utils as utils
import cellCnn.downsample as downsample

# Import data and data transformation


In [7]:
fixed_path = '/content/drive/MyDrive/Master Thesis/CellCNN/'
cell_repo = '/content/drive/MyDrive//Master Thesis/Code_trial_1/B-ALL_expression_matrix.csv'


In [8]:
df, raw_df = read_data(cell_repo)

# transform data to be in the right format
raw_df = pd.read_csv(cell_repo, sep = ';', decimal = ',')
df = raw_df.copy()

### Data converted in a Padas Dataframe ###


In [9]:
df = raw_df.drop(columns = ['Original_ID'])

# check number of healthy and blast cells
blast_cells = (df['IsBlast'] == 1).sum()
healthy_cells = (df['IsBlast'] == 0).sum()
blast_perc = blast_cells / len(df)

print(f'Number of Blast Cells: {blast_cells}')
print(f'Number of Healthy Cells: {healthy_cells}')
print(f'% of Blast Cells: {(blast_perc *100):.5f}%')


Number of Blast Cells: 2390
Number of Healthy Cells: 476610
% of Blast Cells: 0.49896%


#  Datasets Generation, Train, Test and Validation Splitting



- Blast, Healthy cells division

- Artificially creation of 20 Patients cells samples (10 healthy, 10 with blast cells):
  - 20000 cells for each patient
  - For donors with *blast* cells, the chosen % of blast cells is chosen randomly from the blast cells dataset. Then (20000 - blast_cells) healthy cells are sampled from the healthy dataset

- Trining set + validation set-> 70 % of the samples (12 samples for training, 2 for validation )
- Test set -> 30 % of the samples (6 remaining remaining)

Note: all sets are half healthy donor and half donors with blast cells

# From Pandas DF to Array


In [10]:
# Reproducibility test
df1_results = dataset_split(df, seed=42)
df2_results = dataset_split(df, seed=42)
train_dataset_1 = df1_results[0]
train_dataset_2 = df2_results[0]

print(train_dataset_1[0][1:5])
print(train_dataset_2[0][1:5])

           FSC-A     FSC-H     SSC-A  CD81(FITC-A)  CD73+CD304(PE-A)  \
231282  1.166987  1.418838  0.557663      2.184032          1.996026   
105159  0.718468  0.916034  0.460473      2.219524          0.972199   
159675  0.568573  0.655975  0.428337      0.924972          0.534442   
253768  0.765798  0.952707  0.505173      2.154496          2.061495   

        CD34(PerCP-Cy5-5-A)  CD19(PE-Cy7-A)  CD10(APC-A)  CD38(APC-C750-A)  \
231282             1.250122        3.083143     0.699634          1.197508   
105159             0.401445        0.540750     0.230550          1.205658   
159675             0.623103        0.319829     0.544888          0.456514   
253768            -0.008778        1.034054     0.602270          1.066575   

        CD20(PB-A)  CD45(OC515-A)  
231282    3.600320       2.811800  
105159   -0.091524       2.693729  
159675    0.302994       1.121270  
253768    0.134448       2.515873  
           FSC-A     FSC-H     SSC-A  CD81(FITC-A)  CD73+CD304(PE-A)

In [16]:

datasets_trials = []
splitting_param = [(20000, 20), (8000, 40), (3900, 60)]

trials = len(splitting_param)
for i in range(trials):
    train_dataset, train_y, val_dataset, val_y, test_dataset, test_y, all_samples, all_samples_labels = dataset_split(df, splitting_param[i][0], 0.005, splitting_param[i][1], seed = 42) # build train, test and val datasets

    # Train
    train_samples_array = df_to_array(train_dataset)
    train_y_array = np.array(train_y)

    # Validation
    val_samples_array = df_to_array(val_dataset)
    val_y_array = np.array(val_y)

    #Test
    test_samples_array = df_to_array(test_dataset)
    test_y_array = np.array(test_y)

    datasets_i = [train_samples_array, train_y_array, val_samples_array, val_y_array, test_samples_array, test_y_array, all_samples, all_samples_labels]
    datasets_trials.append(datasets_i)


#print(len(datasets_trials[1][0][0]))

# Training Section

In [ ]:
#filters = list(range(20))
#print(filters)

model = CellCnn(
    ncell= 1000,            #200                        # Number of cells per multi-cell input (sampled from the 'patient' datasets)
    nsubset=int(10),#500                             # Total number of multi-cell inputs that will be generated per class (or sample and class)
    per_sample = True,                                # For each sample, nsubset samples of ncell are performed
    nfilter_choice= [3,5,7,9], #list(range(3,21)),                 # Range of possible number of filters
    maxpool_percentages=[0.01, 1., 5., 20., 100.],    # list of k-percentage max_pooling
    learning_rate=0.01,                               # Learning rate
    max_epochs=50, #50
    patience=5,                                       # Early stopping patience
    nrun=15,  #15                                     # Number of neural network configurations to try (for Hyperparameter optimization)
    regression=False,
    scale=True,                                       # Z-score normalization
    verbose=1
)

outdir = '/content/cellcnn_results'  # Results Directory

hy_par_choice = []
for i in range(trials):
    train_samples_array = datasets_trials[i][0]
    train_y_array = datasets_trials[i][1]
    val_samples_array = datasets_trials[i][2]
    val_y_array = datasets_trials[i][3]

    model.fit(
        train_samples=train_samples_array,
        train_phenotypes=train_y_array,
        outdir=outdir,
        valid_samples= val_samples_array,
        valid_phenotypes= val_y_array,
        generate_valid_set = False
    )

    hy_par_choice.append(model.results['config'])


print("Done!")
#print(isinstance(model, str))


# ============================================= #
Run: 0

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5500 - loss: 0.6920 - val_accuracy: 0.5000 - val_loss: 0.6931
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - accuracy: 0.4917 - loss: 0.6950 - val_accuracy: 0.5000 - val_loss: 0.6929
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step - accuracy: 0.5083 - loss: 0.6946 - val_accuracy: 0.5500 - val_loss: 0.6928
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - accuracy: 0.5917 - loss: 0.6904 - val_accuracy: 0.5500 - val_loss: 0.6926
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.5583 - loss: 0.6896 - val_accuracy: 0.5000 - val_loss: 0.6923
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.6333 - loss: 0.6830 - val_accuracy: 0.6000 - val_loss: 0.6920
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.5250 - loss: 0.6906 - val_accuracy: 0.6000 - val_loss: 0.6916
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 

In [18]:

total_trial_parametes = show_hyperparameters(hy_par_choice)

#print(isinstance(model, str))

Model 1 -> nfilter: 5, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 9, learning_rate: 0, maxpool_percentage: 5.0
Model 1 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 9, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 3, learning_rate: 0, maxpool_percentage: 5.0
Model 1 -> nfilter: 5, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 3, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 5.0


#Model Testing

In [19]:

predictions = []
predictions_str = []
for i in range(trials):
    test_samples_array = datasets_trials[i][4]
    #print(isinstance(test_samples_array, list))

    test_y_array = datasets_trials[i][5]

    test_pred = model.predict(test_samples_array)
    predictions.append(test_pred)

    # print results
    pred_phenotypes_array = np.array(phenotype_prediction(test_pred)) # extract predictions

    comparison = pred_phenotypes_array ==  test_y_array #checks differencies in prediction
    accuracy = np.sum(comparison)/ len(test_y_array)  #compute accuracy

    pred_str = ''

    pred_str += f'Predicted Phenotypes: {pred_phenotypes_array} -> True phenotypes: {test_y_array}\n'
    pred_str += f'Trial Accuracy: {accuracy}'

    predictions_str.append(pred_str)




1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


In [20]:

for i, pred_str in enumerate(predictions_str):
    print(f'\nTrial {i+1}')


    trial_par = total_trial_parametes[i]
    print(f'len trial_par: {len(trial_par)}')
    for model in trial_par:
        print(model)

    print(pred_str)


Trial 1
len trial_par: 3
Model 1 -> nfilter: 5, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 9, learning_rate: 0, maxpool_percentage: 5.0
Predicted Phenotypes: [1 1 1 1 1 1] -> True phenotypes: [0 1 0 1 0 1]
Trial Accuracy: 0.5

Trial 2
len trial_par: 3
Model 1 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 9, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 3, learning_rate: 0, maxpool_percentage: 5.0
Predicted Phenotypes: [1 1 1 1 1 1 1 1 1 1 1 1] -> True phenotypes: [0 1 0 1 0 1 0 1 0 1 0 1]
Trial Accuracy: 0.5

Trial 3
len trial_par: 3
Model 1 -> nfilter: 5, learning_rate: 0, maxpool_percentage: 0.01 <- Best Model of the trial
Model 2 -> nfilter: 3, learning_rate: 0, maxpool_percentage: 1.0
Model 3 -> nfilter: 7, learning_rate: 0, maxpool_percentage: 5.0
Predicted Phenotypes: [1 1 1 1 1 1 1 1 1 1 1 1 1 

In [18]:
from sklearn.metrics import roc_auc_score
# calculate area under the ROC curve for the test set
test_auc = roc_auc_score(test_y_array_1, test_pred[:,1])
print(test_auc)

ValueError: Found input variables with inconsistent numbers of samples: [6, 12]

# Results Visualization

In [ ]:

markers_label = list(df.columns)
print(markers_label)

_ = plotting.plot_results(model.results, test_samples_array_1, test_y_array_1,
                 markers_label, outdir, filter_response_thres=0,
                 filter_diff_thres=0.2, group_a='healthy', group_b='blast',
                          tsne_ncell = 10000, # Set to 0 to skip t-SNE plotting
                          show_filters=True)

print("Attempting to plot results, skipping t-SNE grid.")

['FSC-A', 'FSC-H', 'SSC-A', 'CD81(FITC-A)', 'CD73+CD304(PE-A)', 'CD34(PerCP-Cy5-5-A)', 'CD19(PE-Cy7-A)', 'CD10(APC-A)', 'CD38(APC-C750-A)', 'CD20(PB-A)', 'CD45(OC515-A)', 'IsBlast']
Showing Filters
(120000, 11)
x shape: 120000
scaler not present False
Computing t-SNE projection...
(120000, 11)
[[ 0.11448655  0.22280259 -0.36375225 ... -0.70846312 -0.30362855
   0.63908024]
 [ 0.66029979  0.6871535   0.28931255 ...  1.17250529 -0.82384817
   0.7076581 ]
 [ 0.54906101  0.39641056 -0.10758904 ... -1.1795732  -0.03410582
  -1.31042402]
 ...
 [-0.85774369 -0.9475321  -0.50319383 ...  0.53915726 -0.08358513
  -1.87113767]
 [-1.21728588 -1.27995439 -0.51534632 ...  2.02992784 -0.82072047
  -1.228664  ]
 [-1.13483807 -1.22847987 -0.43863667 ...  1.54174052  1.2311125
  -0.64204996]]
x_for_tsne: [96816 71032 55865 ... 85480 49333 96314]
10000
Using perplexity: 30 for 10000 samples
DEBUG g_i: 2 <class 'int'>
DEBUG g_j: 7 <class 'int'>
DEBUG ncol: 11 <class 'int'>
True z deve essere numerico

Tru

In [ ]:
'''

"""
Implementazione di CITRUS per classificazione di dati di citometria a flusso
CITRUS: Cluster Identification, Characterization, and Regression

Dati: Sets di cellule con marcatori, ogni set ha fenotipo (sano/malato)
"""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

class CITRUS:
    """
    Implementazione del modello CITRUS per classificazione
    """

    def __init__(self,
                 n_clusters_range=(5, 50),
                 min_cluster_size=100,
                 feature_selection_method='regularized_regression',
                 classifier='logistic',
                 cross_validation_folds=5,
                 random_state=42):
        """
        Parametri:
        - n_clusters_range: range per il numero di cluster da testare
        - min_cluster_size: dimensione minima del cluster
        - feature_selection_method: metodo per selezione features
        - classifier: classificatore da usare ('logistic', 'random_forest')
        - cross_validation_folds: fold per cross-validation
        """
        self.n_clusters_range = n_clusters_range
        self.min_cluster_size = min_cluster_size
        self.feature_selection_method = feature_selection_method
        self.classifier = classifier
        self.cv_folds = cross_validation_folds
        self.random_state = random_state

        # Attributi che verranno popolati durante il training
        self.scaler = None
        self.optimal_n_clusters = None
        self.cluster_model = None
        self.selected_features = None
        self.final_classifier = None
        self.cluster_frequencies = None
        self.feature_names = None

    def preprocess_data(self, cell_data_list, phenotypes, marker_names):
        """
        Preprocessa i dati per CITRUS

        Args:
        - cell_data_list: lista di array numpy, ogni array è un set di cellule
        - phenotypes: lista di etichette (0=sano, 1=malato)
        - marker_names: lista con nomi dei marcatori

        Returns:
        - combined_data: dati combinati di tutte le cellule
        - sample_labels: etichette per ogni cellula indicando il campione di provenienza
        """
        print("Preprocessing dei dati...")

        # Combina tutti i dati cellulari
        combined_data = []
        sample_labels = []

        for i, cell_data in enumerate(cell_data_list):
            combined_data.append(cell_data)
            sample_labels.extend([i] * len(cell_data))

        combined_data = np.vstack(combined_data)
        sample_labels = np.array(sample_labels)

        # Standardizzazione
        self.scaler = StandardScaler()
        combined_data_scaled = self.scaler.fit_transform(combined_data)

        # Salva i nomi dei marcatori
        self.feature_names = marker_names

        print(f"Dati preprocessati: {combined_data_scaled.shape[0]} cellule, {combined_data_scaled.shape[1]} marcatori")
        return combined_data_scaled, sample_labels, phenotypes

    def find_optimal_clusters(self, data):
        """
        Trova il numero ottimale di cluster usando il metodo del gomito
        """
        print("Ricerca del numero ottimale di cluster...")

        inertias = []
        k_range = range(self.n_clusters_range[0], self.n_clusters_range[1] + 1, 5)

        for k in k_range:
            kmeans = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
            kmeans.fit(data)
            inertias.append(kmeans.inertia_)

        # Calcola la derivata seconda per trovare il gomito
        if len(inertias) >= 3:
            second_derivative = np.diff(inertias, 2)
            optimal_idx = np.argmax(second_derivative) + 2
            self.optimal_n_clusters = k_range[optimal_idx]
        else:
            self.optimal_n_clusters = k_range[len(k_range)//2]

        print(f"Numero ottimale di cluster: {self.optimal_n_clusters}")
        return self.optimal_n_clusters

    def perform_clustering(self, data):
        """
        Esegue il clustering sui dati
        """
        print("Esecuzione del clustering...")

        # Trova numero ottimale di cluster se non specificato
        if self.optimal_n_clusters is None:
            self.find_optimal_clusters(data)

        # Esegui K-means
        self.cluster_model = KMeans(
            n_clusters=self.optimal_n_clusters,
            random_state=self.random_state,
            n_init=10
        )
        cluster_labels = self.cluster_model.fit_predict(data)

        print(f"Clustering completato con {self.optimal_n_clusters} cluster")
        return cluster_labels

    def calculate_cluster_frequencies(self, cluster_labels, sample_labels, phenotypes):
        """
        Calcola le frequenze dei cluster per ogni campione
        """
        print("Calcolo delle frequenze dei cluster...")

        n_samples = len(set(sample_labels))
        n_clusters = self.optimal_n_clusters

        # Matrice delle frequenze: righe=campioni, colonne=cluster
        cluster_freq_matrix = np.zeros((n_samples, n_clusters))

        for sample_idx in range(n_samples):
            # Cellule del campione corrente
            sample_mask = sample_labels == sample_idx
            sample_clusters = cluster_labels[sample_mask]

            # Calcola frequenze relative per ogni cluster
            for cluster_idx in range(n_clusters):
                cluster_count = np.sum(sample_clusters == cluster_idx)
                total_cells = len(sample_clusters)
                cluster_freq_matrix[sample_idx, cluster_idx] = cluster_count / total_cells

        self.cluster_frequencies = cluster_freq_matrix
        return cluster_freq_matrix, np.array(phenotypes)

    def select_discriminative_features(self, features, labels):
        """
        Seleziona le features più discriminative usando regressione regolarizzata
        """
        print("Selezione delle features discriminative...")

        selected_indices = None

        if self.feature_selection_method == 'regularized_regression':
            selector = LogisticRegression(
                penalty='l1',
                solver='liblinear',
                random_state=self.random_state,
                C=0.1
            )
            selector.fit(features, labels)
            feature_importance = np.abs(selector.coef_[0])
            threshold = np.percentile(feature_importance, 75)  # Top 25%
            selected_indices = feature_importance > threshold

        elif self.feature_selection_method == 'statistical':
            p_values = []
            for i in range(features.shape[1]):
                group_0 = features[labels == 0, i]
                group_1 = features[labels == 1, i]
                _, p_val = stats.mannwhitneyu(group_0, group_1)
                p_values.append(p_val)
            selected_indices = np.array(p_values) < 0.05

        # 🔴 PATCH: se nessuna feature selezionata, fallback a tutte
        if selected_indices is None or not np.any(selected_indices):
            print("⚠️ Nessuna feature discriminativa trovata, uso tutte le features come fallback.")
            selected_indices = np.ones(features.shape[1], dtype=bool)

        self.selected_features = selected_indices
        n_selected = np.sum(selected_indices)
        print(f"Selezionate {n_selected} features su {features.shape[1]} totali")

        return features[:, selected_indices]

    def train_classifier(self, features, labels):
        """
        Addestra il classificatore finale
        """
        print("Addestramento del classificatore finale...")

        if self.classifier == 'logistic':
            self.final_classifier = LogisticRegression(
                random_state=self.random_state,
                max_iter=1000
            )
        elif self.classifier == 'random_forest':
            self.final_classifier = RandomForestClassifier(
                n_estimators=100,
                random_state=self.random_state
            )

        # Cross-validation
        cv_scores = cross_val_score(
            self.final_classifier, features, labels,
            cv=self.cv_folds, scoring='accuracy'
        )

        print(f"Accuratezza cross-validation: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

        # Addestramento finale
        self.final_classifier.fit(features, labels)

        return cv_scores

    def fit(self, cell_data_list, phenotypes, marker_names):
        """
        Metodo principale per addestrare CITRUS
        """
        print("=== Inizio addestramento CITRUS ===")

        # 1. Preprocessing
        data_scaled, sample_labels, phenotype_labels = self.preprocess_data(
            cell_data_list, phenotypes, marker_names
        )

        # 2. Clustering
        cluster_labels = self.perform_clustering(data_scaled)

        # 3. Calcolo frequenze cluster
        cluster_features, sample_phenotypes = self.calculate_cluster_frequencies(
            cluster_labels, sample_labels, phenotype_labels
        )

        # 4. Selezione features discriminative
        selected_features = self.select_discriminative_features(
            cluster_features, sample_phenotypes
        )

        # 5. Addestramento classificatore
        cv_scores = self.train_classifier(selected_features, sample_phenotypes)

        print("=== Addestramento CITRUS completato ===")
        return cv_scores

    def predict(self, new_cell_data_list):
        """
        Predice le etichette per nuovi dati
        """
        if self.final_classifier is None:
            raise ValueError("Modello non ancora addestrato. Chiama prima fit()")

        print("Predizione su nuovi dati...")

        # Preprocessing dei nuovi dati
        combined_data = np.vstack(new_cell_data_list)
        data_scaled = self.scaler.transform(combined_data)

        # Clustering
        cluster_labels = self.cluster_model.predict(data_scaled)

        # Calcolo frequenze per ogni nuovo campione
        n_samples = len(new_cell_data_list)
        cluster_freq_matrix = np.zeros((n_samples, self.optimal_n_clusters))

        start_idx = 0
        for i, cell_data in enumerate(new_cell_data_list):
            end_idx = start_idx + len(cell_data)
            sample_clusters = cluster_labels[start_idx:end_idx]

            for cluster_idx in range(self.optimal_n_clusters):
                cluster_count = np.sum(sample_clusters == cluster_idx)
                cluster_freq_matrix[i, cluster_idx] = cluster_count / len(cell_data)

            start_idx = end_idx

        # Selezione features e predizione
        selected_features = cluster_freq_matrix[:, self.selected_features]
        predictions = self.final_classifier.predict(selected_features)
        probabilities = self.final_classifier.predict_proba(selected_features)

        return predictions, probabilities

    def plot_results(self, cell_data_list, phenotypes, marker_names):
        """
        Visualizza i risultati dell'analisi CITRUS
        """
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        # 1. Distribuzione dei cluster
        combined_data = np.vstack(cell_data_list)
        data_scaled = self.scaler.transform(combined_data)
        cluster_labels = self.cluster_model.predict(data_scaled)

        # PCA per visualizzazione 2D
        pca = PCA(n_components=2)
        data_pca = pca.fit_transform(data_scaled)

        axes[0,0].scatter(data_pca[:, 0], data_pca[:, 1], c=cluster_labels, cmap='tab20', alpha=0.6, s=1)
        axes[0,0].set_title('Distribuzione dei Cluster (PCA)')
        axes[0,0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
        axes[0,0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')

        # 2. Frequenze cluster per fenotipo
        cluster_freq_df = pd.DataFrame(self.cluster_frequencies)
        cluster_freq_df['phenotype'] = phenotypes

        # Heatmap delle frequenze medie per fenotipo
        freq_by_phenotype = cluster_freq_df.groupby('phenotype').mean()
        sns.heatmap(freq_by_phenotype.T, annot=True, fmt='.3f', cmap='viridis', ax=axes[0,1])
        axes[0,1].set_title('Frequenze Cluster per Fenotipo')
        axes[0,1].set_xlabel('Fenotipo')
        axes[0,1].set_ylabel('Cluster')

        # 3. Features selezionate
        selected_clusters = np.where(self.selected_features)[0]
        if len(selected_clusters) > 0:
            selected_freq = self.cluster_frequencies[:, self.selected_features]

            # Boxplot delle features più discriminative
            n_features_to_show = min(6, len(selected_clusters))
            feature_indices = selected_clusters[:n_features_to_show]

            boxplot_data = []
            boxplot_labels = []
            boxplot_phenotypes = []

            for i, feat_idx in enumerate(feature_indices):
                for phenotype in [0, 1]:
                    phenotype_mask = np.array(phenotypes) == phenotype
                    values = self.cluster_frequencies[phenotype_mask, feat_idx]
                    boxplot_data.extend(values)
                    boxplot_labels.extend([f'Cluster {feat_idx}'] * len(values))
                    boxplot_phenotypes.extend(['Sano' if phenotype == 0 else 'Malato'] * len(values))

            boxplot_df = pd.DataFrame({
                'Frequency': boxplot_data,
                'Cluster': boxplot_labels,
                'Phenotype': boxplot_phenotypes
            })

            sns.boxplot(data=boxplot_df, x='Cluster', y='Frequency', hue='Phenotype', ax=axes[1,0])
            axes[1,0].set_title('Frequenze dei Cluster Discriminativi')
            axes[1,0].tick_params(axis='x', rotation=45)

        # 4. Importanza delle features se disponibile
        if hasattr(self.final_classifier, 'coef_'):
            feature_importance = np.abs(self.final_classifier.coef_[0])
            selected_cluster_ids = np.where(self.selected_features)[0]

            importance_df = pd.DataFrame({
                'Cluster': [f'Cluster {i}' for i in selected_cluster_ids],
                'Importance': feature_importance
            }).sort_values('Importance', ascending=True)

            axes[1,1].barh(importance_df['Cluster'], importance_df['Importance'])
            axes[1,1].set_title('Importanza dei Cluster nel Classificatore')
            axes[1,1].set_xlabel('Importanza (|coefficiente|)')

        plt.tight_layout()
        plt.show()



print("=== ESEMPIO DI UTILIZZO CITRUS CON DATI REALI ===")

# Parametri
n_samples = 200 # Numero di cellule per campione (riduci se necessario)
markers_labels = list(df.columns)
n_markers = len(markers_labels)

print(f"Elaborazione {len(all_samples)} campioni con {n_markers} marcatori")

# Prepara i dati
samples_to_process = []
for i, sample in enumerate(all_samples):
    # Campionamento sicuro
    sample_size = min(n_samples, len(sample))
    if sample_size < 100:  # Salta campioni troppo piccoli
        print(f"Campione {i} troppo piccolo ({sample_size} cellule), saltato")
        continue

    sample_data = sample.sample(sample_size, replace=False)
    # Converti in numpy array e gestisci eventuali NaN
    sample_array = sample_data.values
    if np.any(np.isnan(sample_array)):
        print(f"Trovati NaN nel campione {i}, rimozione...")
        sample_array = sample_array[~np.isnan(sample_array).any(axis=1)]

    samples_to_process.append(sample_array)

# Aggiusta le etichette per i campioni effettivamente processati
processed_labels = all_samples_labels[:len(samples_to_process)]

print(f"Campioni processati: {len(samples_to_process)}")
print(f"Distribuzione etichette: {np.bincount(processed_labels)}")

# Inizializza CITRUS
citrus_model = CITRUS(
    n_clusters_range=(10, 30),
    min_cluster_size=50,
    classifier='logistic',
    random_state=42
)

try:
    # Addestramento
    cv_scores = citrus_model.fit(samples_to_process, processed_labels, markers_labels)

    # Visualizzazione risultati
    citrus_model.plot_results(samples_to_process, processed_labels, markers_labels)

    print(f"Accuratezza cross-validation: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

except Exception as e:
    print(f"Errore durante l'addestramento: {e}")
    import traceback
    traceback.print_exc()

'''